# Моделирование динамики информационных событий через Hawkes-процесс

Этот ноутбук реализует **оба варианта** для диплома:

1. **Вариант 1: синтетика**  
   Генерируем каскады Hawkes-процесса, затем проверяем, насколько хорошо модель восстанавливает параметры `mu`, `alpha`, `decay`.

2. **Вариант 2: реальный датасет**  
   Для каждого информационного события (кластера) берём времена публикаций, обучаем одномерный Hawkes-процесс и строим прогноз по первым `N` сообщениям.

## Что именно оцениваем

- `mu` — фоновая интенсивность.
- `alpha` — сила самовозбуждения.
- `decay` — скорость затухания влияния прошлых сообщений.

## Метрики

- `RMSE` числа сообщений в ближайший час.
- `RMSE` итогового размера каскада на фиксированном горизонте.
- Корреляция между `alpha` и фактическим размером каскада.

## Важное замечание по окружению

Изначально задача планировалась на `tick`, но в актуальных окружениях у `tick` есть практические проблемы совместимости и баги learner-классов. В вашем случае `HawkesExpKern` падает уже в конструкторе с ошибкой `AttributeError: 'HawkesExpKern' object has no settable attribute 'events'`, то есть проблема находится внутри библиотеки.

Поэтому ноутбук ниже использует **самодостаточную реализацию одномерного Hawkes-процесса** на `numpy + scipy`, без зависимости от `tick` для оценки параметров. Для диплома это даже надёжнее: все формулы и оценка полностью воспроизводимы.

Для реальных данных в ноутбуке используется **фиксированный горизонт прогноза**, чтобы избежать утечки данных. Если выбран горизонт `24 часа`, то в оценку попадают только те каскады, которые реально наблюдались не меньше 24 часов от первого сообщения.

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -U pip setuptools wheel
pip install -r requirements.txt
jupyter notebook
```

## Архитектура решения

### 1. Подготовка данных

На вход нужен датасет с минимум двумя полями:

- `cluster_id` — идентификатор информационного события.
- `published_at` — время публикации сообщения.

Если в датасете **нет готовых кластеров**, Hawkes-модель обучать рано: сначала нужен отдельный этап кластеризации новостей по событию.

### 2. Построение каскадов

Для каждого `cluster_id`:

- сортируем публикации по времени;
- переводим время в секунды;
- сдвигаем начало каскада в `t = 0`.

### 3. Обучение модели

Для одного кластера используем одномерный Hawkes-процесс с экспоненциальным ядром:

$$
\lambda(t) = \mu + \sum_{t_i < t} \alpha \cdot \beta \cdot e^{-\beta (t - t_i)}
$$

где:

- `mu` — фон,
- `alpha` — сила возбуждения,
- `beta = decay` — скорость затухания.

### 4. Оценка `decay`

Практический нюанс: в `tick` класс `HawkesExpKern` обычно обучает `mu` и `alpha` **при фиксированном `decay`**. Поэтому в ноутбуке `decay` выбирается через **поиск по сетке**: пробуем несколько значений и берём лучшее по логарифму правдоподобия.

### 5. Прогноз по первым `N` сообщениям

Берём только префикс каскада:

- первые `N` сообщений;
- обучаем `mu`, `alpha`, `decay`;
- прогнозируем:
  - сколько сообщений придёт в ближайший час;
  - каков будет итоговый размер каскада в окне наблюдения.

### 6. Оценка качества

По набору каскадов считаем:

- `RMSE(next_hour_count)`
- `RMSE(final_size)`
- `corr(alpha, final_size)`

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
LARGE_PENALTY = 1e12


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_rmse(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    return rmse(y_true, y_pred)


def safe_corr(x, y):
    if len(x) < 2 or len(y) < 2:
        return np.nan
    return float(pd.Series(x).corr(pd.Series(y)))


def ensure_strictly_increasing(timestamps, eps=1e-6):
    """
    В реальных данных несколько постов могут иметь один и тот же timestamp.
    Для непрерывного Hawkes-процесса делаем минимальный детерминированный сдвиг.
    """
    ts = np.sort(np.asarray(timestamps, dtype=float)).copy()
    if len(ts) == 0:
        return ts
    for i in range(1, len(ts)):
        if ts[i] <= ts[i - 1]:
            ts[i] = ts[i - 1] + eps
    return ts


def hawkes_recursive_sum(timestamps, decay):
    """
    R[i] = sum_{j < i} exp(-decay * (t_i - t_j))
    для одномерного Hawkes-процесса с экспоненциальным ядром.
    """
    ts = np.asarray(timestamps, dtype=float)
    n = len(ts)
    r = np.zeros(n, dtype=float)
    for i in range(1, n):
        dt = max(ts[i] - ts[i - 1], 0.0)
        r[i] = math.exp(-decay * dt) * (1.0 + r[i - 1])
    return r


def neg_loglik_univariate_exp(params, timestamps, end_time=None):
    """
    Negative log-likelihood for
    lambda(t) = mu + alpha * decay * sum exp(-decay * (t - t_j)).
    Возвращает конечное штрафное значение вместо inf, чтобы scipy не ломал numdiff.
    """
    mu, alpha, decay = params
    ts = ensure_strictly_increasing(timestamps)

    if len(ts) < 2:
        return LARGE_PENALTY
    if not np.all(np.isfinite([mu, alpha, decay])):
        return LARGE_PENALTY
    if mu <= 0 or alpha < 0 or alpha >= 0.999 or decay <= 0:
        return LARGE_PENALTY

    if end_time is None:
        end_time = float(ts[-1] + 1e-6)
    end_time = max(float(end_time), float(ts[-1] + 1e-6))

    r = hawkes_recursive_sum(ts, decay)
    intensities = mu + alpha * decay * r
    if np.any(~np.isfinite(intensities)) or np.any(intensities <= 1e-12):
        return LARGE_PENALTY

    tail = end_time - ts
    if np.any(tail < 0):
        return LARGE_PENALTY

    compensator = mu * end_time + alpha * np.sum(1.0 - np.exp(-decay * tail))
    loglik = np.sum(np.log(intensities)) - compensator
    if not np.isfinite(loglik):
        return LARGE_PENALTY
    return float(-loglik)


def _build_initial_guesses(ts, end_time):
    diffs = np.diff(ts)
    diffs = diffs[diffs > 1e-9]
    typical_gap = float(np.median(diffs)) if len(diffs) else max(end_time / max(len(ts), 1), 1.0)
    naive_rate = len(ts) / max(end_time, 1e-6)
    decay_center = 1.0 / max(typical_gap, 1.0)

    return [
        np.array([max(naive_rate * 0.2, 1e-5), 0.20, max(decay_center * 0.5, 1e-5)], dtype=float),
        np.array([max(naive_rate * 0.4, 1e-5), 0.40, max(decay_center, 1e-5)], dtype=float),
        np.array([max(naive_rate * 0.6, 1e-5), 0.70, max(decay_center * 2.0, 1e-5)], dtype=float),
    ]


def fit_hawkes_mle(timestamps, end_time=None, init=None):
    ts = ensure_strictly_increasing(timestamps)

    if len(ts) < 2:
        raise ValueError("Для оценки Hawkes-процесса нужно минимум 2 события.")

    if end_time is None:
        end_time = float(ts[-1] + 1e-6)
    end_time = max(float(end_time), float(ts[-1] + 1e-6))

    naive_rate = len(ts) / max(end_time, 1e-6)
    bounds = [
        (1e-8, max(naive_rate * 20, 5.0)),
        (1e-8, 0.999),
        (1e-8, 10.0),
    ]

    starts = [init] if init is not None else _build_initial_guesses(ts, end_time)
    best_result = None

    for x0 in starts:
        result = minimize(
            neg_loglik_univariate_exp,
            x0=np.asarray(x0, dtype=float),
            args=(ts, end_time),
            method="Powell",
            bounds=bounds,
            options={"maxiter": 400, "xtol": 1e-4, "ftol": 1e-4},
        )
        if best_result is None or result.fun < best_result.fun:
            best_result = result

    if best_result is None or not np.isfinite(best_result.fun):
        raise RuntimeError("Оптимизация Hawkes MLE не вернула конечное значение функции.")

    refine = minimize(
        neg_loglik_univariate_exp,
        x0=best_result.x,
        args=(ts, end_time),
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": 300},
    )
    if np.isfinite(refine.fun) and refine.fun <= best_result.fun:
        best_result = refine

    mu, alpha, decay = best_result.x
    if not np.all(np.isfinite([mu, alpha, decay])):
        raise RuntimeError("Оценка Hawkes MLE дала нечисловые параметры.")

    return {
        "mu": float(mu),
        "alpha": float(alpha),
        "decay": float(decay),
        "score": float(-best_result.fun),
        "optimizer_message": best_result.message,
        "n_iter": int(getattr(best_result, "nit", 0) or 0),
    }


def excitation_state(history, decay, t_ref=None):
    history = ensure_strictly_increasing(history)
    if len(history) == 0:
        return 0.0
    if t_ref is None:
        t_ref = float(history[-1])
    return float(np.exp(-decay * (t_ref - history[history <= t_ref])).sum())


def hawkes_intensity(mu, alpha, decay, history, t):
    history = ensure_strictly_increasing(history)
    past = history[history < t]
    if len(past) == 0:
        return float(mu)
    return float(mu + alpha * decay * np.exp(-decay * (t - past)).sum())


def intensity_curve(mu, alpha, decay, history, grid):
    return np.array([hawkes_intensity(mu, alpha, decay, history, t) for t in grid], dtype=float)


def simulate_univariate_hawkes(mu, alpha, decay, horizon_sec, history=None, seed=42):
    rng_local = np.random.default_rng(seed)
    history = ensure_strictly_increasing([] if history is None else history)

    if len(history) > 0:
        t0 = float(history[-1])
        g = excitation_state(history, decay, t_ref=t0)
    else:
        t0 = 0.0
        g = 0.0

    t = 0.0
    current_lambda = mu + alpha * decay * g
    new_events = []

    while t < horizon_sec:
        lambda_bar = max(current_lambda, mu + 1e-12)
        wait = -math.log(rng_local.uniform()) / lambda_bar
        t += wait
        if t > horizon_sec:
            break

        g = g * math.exp(-decay * wait)
        lambda_t = mu + alpha * decay * g

        if rng_local.uniform() <= lambda_t / lambda_bar:
            event_time = t0 + t
            new_events.append(event_time)
            g += 1.0
            current_lambda = mu + alpha * decay * g
        else:
            current_lambda = lambda_t

    return np.asarray(new_events, dtype=float)


def simulate_future_counts(mu, alpha, decay, history, horizon_sec, n_simulations=500, seed=42):
    history = ensure_strictly_increasing(history)
    if len(history) == 0:
        raise ValueError("История событий не может быть пустой.")

    counts = []
    for sim_id in range(n_simulations):
        future_events = simulate_univariate_hawkes(
            mu=mu,
            alpha=alpha,
            decay=decay,
            horizon_sec=horizon_sec,
            history=history,
            seed=seed + sim_id,
        )
        counts.append(len(future_events))

    return np.asarray(counts, dtype=int)


def actual_future_count(full_timestamps, prefix_size, horizon_sec):
    ts = ensure_strictly_increasing(full_timestamps)
    cutoff = ts[prefix_size - 1]
    return int(((ts > cutoff) & (ts <= cutoff + horizon_sec)).sum())


def actual_total_count_at_horizon(full_timestamps, total_horizon_sec):
    ts = ensure_strictly_increasing(full_timestamps)
    return int((ts <= total_horizon_sec).sum())


def evaluate_prefix_forecast(full_timestamps, prefix_size, total_horizon_sec, hour_horizon=3600, n_simulations=500, seed=42):
    ts = ensure_strictly_increasing(full_timestamps)
    if len(ts) <= prefix_size:
        raise ValueError("Каскад слишком короткий для выбранного prefix_size.")
    if ts[-1] < total_horizon_sec:
        raise ValueError("Каскад наблюдался меньше, чем выбранный фиксированный горизонт оценки.")

    prefix_abs = ts[:prefix_size]
    prefix_rel = prefix_abs - prefix_abs[0]
    prefix_cutoff = float(prefix_rel[-1])
    if prefix_cutoff >= total_horizon_sec:
        raise ValueError("Префикс уже выходит за пределы фиксированного горизонта оценки.")

    fit = fit_hawkes_mle(prefix_rel)
    remaining_horizon = float(total_horizon_sec - prefix_cutoff)

    future_hour_samples = simulate_future_counts(
        mu=fit["mu"],
        alpha=fit["alpha"],
        decay=fit["decay"],
        history=prefix_rel,
        horizon_sec=hour_horizon,
        n_simulations=n_simulations,
        seed=seed,
    )

    future_total_samples = simulate_future_counts(
        mu=fit["mu"],
        alpha=fit["alpha"],
        decay=fit["decay"],
        history=prefix_rel,
        horizon_sec=max(remaining_horizon, 1.0),
        n_simulations=n_simulations,
        seed=seed + 1000,
    )

    return {
        "prefix_size": prefix_size,
        "mu": fit["mu"],
        "alpha": fit["alpha"],
        "decay": fit["decay"],
        "score": fit["score"],
        "actual_next_hour": actual_future_count(ts, prefix_size, hour_horizon),
        "pred_next_hour": float(future_hour_samples.mean()),
        "actual_total": actual_total_count_at_horizon(ts, total_horizon_sec),
        "pred_total": float(prefix_size + future_total_samples.mean()),
        "actual_future_total": int(actual_total_count_at_horizon(ts, total_horizon_sec) - prefix_size),
        "pred_future_total": float(future_total_samples.mean()),
        "evaluation_horizon_sec": float(total_horizon_sec),
    }


def simulate_synthetic_cascade(mu, alpha, decay, end_time_sec, seed=None):
    return simulate_univariate_hawkes(
        mu=mu,
        alpha=alpha,
        decay=decay,
        horizon_sec=end_time_sec,
        history=None,
        seed=seed if seed is not None else RANDOM_SEED,
    )


def generate_synthetic_dataset(
    n_cascades=25,
    end_time_sec=24 * 3600,
    min_events=30,
    max_attempts=30,
    seed=42,
):
    rng_local = np.random.default_rng(seed)
    cascades = []

    for cascade_id in range(n_cascades):
        for attempt in range(max_attempts):
            mu = rng_local.uniform(2e-4, 10e-4)
            alpha = rng_local.uniform(0.15, 0.85)
            decay = rng_local.uniform(1 / 14400, 1 / 1200)
            ts = simulate_synthetic_cascade(mu, alpha, decay, end_time_sec=end_time_sec, seed=seed + cascade_id * 100 + attempt)
            if len(ts) >= min_events:
                cascades.append({
                    "cascade_id": cascade_id,
                    "timestamps": ts,
                    "mu_true": mu,
                    "alpha_true": alpha,
                    "decay_true": decay,
                    "n_events": int(len(ts)),
                    "duration_sec": float(ts[-1] - ts[0]) if len(ts) > 1 else 0.0,
                })
                break
        else:
            warnings.warn(f"Не удалось сгенерировать каскад {cascade_id} с минимум {min_events} событиями.")

    return cascades


def resolve_hf_path(path):
    # Expected form: hf://datasets/<repo_id>/<filename>
    raw = str(path)
    prefix = "hf://datasets/"
    if not raw.startswith(prefix):
        return raw
    rest = raw[len(prefix):]
    parts = rest.split("/")
    if len(parts) < 3:
        raise ValueError(f"Некорректный hf-path: {raw}")
    repo_id = "/".join(parts[:2])
    filename = "/".join(parts[2:])
    return hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")


def load_dataset(path):
    resolved = resolve_hf_path(path)
    suffix = Path(resolved).suffix.lower()

    if suffix in {".json", ".jsonl"}:
        return pd.read_json(resolved, lines=True)
    if suffix == ".csv":
        return pd.read_csv(resolved)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(resolved)

    raise ValueError(f"Неподдерживаемый формат файла: {resolved}")


def _parse_time_column(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() > 0.95:
        sample = float(numeric.dropna().median()) if numeric.dropna().size else np.nan
        if np.isfinite(sample):
            if sample > 1e17:
                return pd.to_datetime(numeric, utc=True, errors="coerce", unit="ns")
            if sample > 1e14:
                return pd.to_datetime(numeric, utc=True, errors="coerce", unit="us")
            if sample > 1e11:
                return pd.to_datetime(numeric, utc=True, errors="coerce", unit="ms")
            if sample > 1e8:
                return pd.to_datetime(numeric, utc=True, errors="coerce", unit="s")
    return pd.to_datetime(series, utc=True, errors="coerce")


def prepare_cascades(df, cluster_col, time_col, min_posts=10, max_cascades=None):
    work = df[[cluster_col, time_col]].copy()
    work[time_col] = _parse_time_column(work[time_col])
    work = work.dropna(subset=[cluster_col, time_col]).sort_values([cluster_col, time_col])

    cascades = []
    for cluster_id, group in work.groupby(cluster_col):
        timestamps_sec = group[time_col].astype("int64").to_numpy() / 1e9
        if len(timestamps_sec) < min_posts:
            continue
        timestamps_rel = timestamps_sec - timestamps_sec[0]
        timestamps_rel = ensure_strictly_increasing(timestamps_rel)
        cascades.append({
            "cluster_id": cluster_id,
            "timestamps": timestamps_rel,
            "n_posts": int(len(timestamps_rel)),
            "duration_sec": float(timestamps_rel[-1]) if len(timestamps_rel) else 0.0,
        })

    cascades = sorted(cascades, key=lambda x: x["n_posts"], reverse=True)
    if max_cascades is not None:
        cascades = cascades[:max_cascades]
    return cascades


def evaluate_cascades(cascades, prefix_size, total_horizon_sec, hour_horizon=3600, n_simulations=500, seed=42):
    rows = []
    for idx, cascade in enumerate(cascades):
        ts = ensure_strictly_increasing(cascade["timestamps"])
        if len(ts) <= prefix_size:
            continue
        if ts[-1] < total_horizon_sec:
            continue
        try:
            metrics = evaluate_prefix_forecast(
                full_timestamps=ts,
                prefix_size=prefix_size,
                total_horizon_sec=total_horizon_sec,
                hour_horizon=hour_horizon,
                n_simulations=n_simulations,
                seed=seed + idx,
            )
            rows.append({**cascade, **metrics})
        except Exception as exc:
            warnings.warn(f"Каскад {cascade.get('cluster_id', cascade.get('cascade_id', idx))} пропущен: {exc}")

    return pd.DataFrame(rows)

## Вариант 1. Синтетика

Цель этого блока: показать, что на контролируемых данных модель восстанавливает параметры и даёт разумный прогноз.

Логика:

1. Генерируем набор искусственных каскадов.
2. Для каждого каскада знаем истинные `mu`, `alpha`, `decay`.
3. Оцениваем параметры через максимум логарифма правдоподобия для одномерного Hawkes-процесса с экспоненциальным ядром.
4. Сравниваем истинные и восстановленные параметры.
5. Проверяем качество прогноза по первым `N` событиям.

In [ ]:
SYNTHETIC_END_TIME_SEC = 24 * 3600
SYNTHETIC_PREFIX_SIZE = 20
SYNTHETIC_HOUR_HORIZON = 3600

synthetic_cascades = generate_synthetic_dataset(
    n_cascades=30,
    end_time_sec=SYNTHETIC_END_TIME_SEC,
    min_events=35,
    seed=RANDOM_SEED,
)

len(synthetic_cascades), synthetic_cascades[0]["n_events"]

In [ ]:
synthetic_recovery_rows = []
for cascade in synthetic_cascades:
    fit = fit_hawkes_mle(cascade["timestamps"])
    synthetic_recovery_rows.append({
        "cascade_id": cascade["cascade_id"],
        "n_events": cascade["n_events"],
        "mu_true": cascade["mu_true"],
        "mu_hat": fit["mu"],
        "alpha_true": cascade["alpha_true"],
        "alpha_hat": fit["alpha"],
        "decay_true": cascade["decay_true"],
        "decay_hat": fit["decay"],
        "score": fit["score"],
    })

synthetic_recovery_df = pd.DataFrame(synthetic_recovery_rows)

summary_recovery = pd.DataFrame({
    "metric": ["RMSE(mu)", "RMSE(alpha)", "RMSE(decay)"],
    "value": [
        rmse(synthetic_recovery_df["mu_true"], synthetic_recovery_df["mu_hat"]),
        rmse(synthetic_recovery_df["alpha_true"], synthetic_recovery_df["alpha_hat"]),
        rmse(synthetic_recovery_df["decay_true"], synthetic_recovery_df["decay_hat"]),
    ],
})

summary_recovery

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, true_col, pred_col, title in [
    (axes[0], "mu_true", "mu_hat", "mu: истинное vs оценка"),
    (axes[1], "alpha_true", "alpha_hat", "alpha: истинное vs оценка"),
    (axes[2], "decay_true", "decay_hat", "decay: истинное vs оценка"),
]:
    ax.scatter(synthetic_recovery_df[true_col], synthetic_recovery_df[pred_col], alpha=0.8)
    line_min = min(synthetic_recovery_df[true_col].min(), synthetic_recovery_df[pred_col].min())
    line_max = max(synthetic_recovery_df[true_col].max(), synthetic_recovery_df[pred_col].max())
    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black")
    ax.set_title(title)
    ax.set_xlabel("Истинное значение")
    ax.set_ylabel("Оценка модели")

plt.tight_layout()
plt.show()

In [ ]:
synthetic_eval_df = evaluate_cascades(
    cascades=synthetic_cascades,
    prefix_size=SYNTHETIC_PREFIX_SIZE,
    hour_horizon=SYNTHETIC_HOUR_HORIZON,
    n_simulations=250,
    seed=RANDOM_SEED,
)

synthetic_metrics = pd.DataFrame({
    "metric": [
        "RMSE(next_hour_count)",
        "RMSE(final_size)",
        "corr(alpha, final_size)",
    ],
    "value": [
        rmse(synthetic_eval_df["actual_next_hour"], synthetic_eval_df["pred_next_hour"]),
        rmse(synthetic_eval_df["actual_total"], synthetic_eval_df["pred_total"]),
        float(synthetic_eval_df["alpha"].corr(synthetic_eval_df["actual_total"])),
    ],
})

synthetic_metrics

In [ ]:
example = synthetic_cascades[np.argmax([c["n_events"] for c in synthetic_cascades])]
example_ts = example["timestamps"]
example_prefix = example_ts[:SYNTHETIC_PREFIX_SIZE]
example_fit = fit_hawkes_mle(example_prefix)

grid = np.linspace(0, min(example_ts[-1], example_prefix[-1] + 6 * 3600), 500)
intensity_vals = intensity_curve(
    mu=example_fit["mu"],
    alpha=example_fit["alpha"],
    decay=example_fit["decay"],
    history=example_prefix,
    grid=grid,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].step(example_ts / 3600, np.arange(1, len(example_ts) + 1), where="post")
axes[0].axvline(example_prefix[-1] / 3600, color="red", linestyle="--", label=f"prefix N={SYNTHETIC_PREFIX_SIZE}")
axes[0].set_title("Синтетический каскад: накопленное число событий")
axes[0].set_xlabel("Время, часы")
axes[0].set_ylabel("Число событий")
axes[0].legend()

axes[1].plot(grid / 3600, intensity_vals, color="darkgreen")
axes[1].vlines(example_prefix / 3600, ymin=0, ymax=intensity_vals.max(), color="gray", alpha=0.2)
axes[1].set_title("Оценённая интенсивность по первым N событиям")
axes[1].set_xlabel("Время, часы")
axes[1].set_ylabel("Интенсивность")

plt.tight_layout()
plt.show()

## Вариант 2. Реальный датасет

Здесь предполагается, что у вас уже есть таблица с публикациями и привязкой каждой публикации к событию (`cluster_id`).

### Минимально необходимые поля

- `cluster_id` — номер события / кластера.
- `published_at` или аналогичное поле времени публикации.

### Если у вас другой нейминг колонок

Просто поменяйте значения в конфиге ниже.

### Если у вас ещё нет кластеров

Это отдельная задача. Для диплома нужно явно формулировать границу метода:

- сначала новости группируются в информационные события внешним методом;
- затем для каждого события моделируется динамика публикаций через Hawkes-процесс.

### Важное методологическое решение

Для честной оценки прогноза ноутбук использует **фиксированный горизонт** `REAL_TOTAL_HORIZON_SEC`.
Например, если это `24 * 3600`, то модель предсказывает размер каскада через 24 часа от первого сообщения.
В метрики попадают только те каскады, которые реально наблюдались не меньше этого горизонта.

In [ ]:
# Укажите здесь свой путь к реальным данным.
# Для ScoutieAutoML/russian-news-telegram-dataset используйте:
# - CLUSTER_COL = "cluster_id"
# - TIME_COL = "create_time"
# create_time в этом датасете хранится как Unix timestamp.

# DATA_PATH = "data/russian_news_clusters.jsonl"
DATA_PATH = "hf://datasets/ScoutieAutoML/russian-news-telegram-dataset/scoutieRussianNewsTelegramDataset.json"
# DATA_PATH = "data/news_clusters.csv"

CLUSTER_COL = "cluster_id"
TIME_COL = "create_time"
MIN_POSTS_PER_CASCADE = 25
MAX_CASCADES = 300
REAL_PREFIX_SIZE = 20
REAL_HOUR_HORIZON = 3600
REAL_TOTAL_HORIZON_SEC = 24 * 3600

In [ ]:
# Запустите эту ячейку после того, как настроите DATA_PATH/CLUSTER_COL/TIME_COL.

real_df = load_dataset(DATA_PATH)
print(f"Исходная таблица: {real_df.shape}")
print("Колонки:", list(real_df.columns))

if CLUSTER_COL not in real_df.columns or TIME_COL not in real_df.columns:
    raise ValueError(
        f"Нужные колонки не найдены. Проверьте CLUSTER_COL={CLUSTER_COL!r} и TIME_COL={TIME_COL!r}."
    )

real_cascades = prepare_cascades(
    df=real_df,
    cluster_col=CLUSTER_COL,
    time_col=TIME_COL,
    min_posts=MIN_POSTS_PER_CASCADE,
    max_cascades=MAX_CASCADES,
)

eligible_cascades = [c for c in real_cascades if c["duration_sec"] >= REAL_TOTAL_HORIZON_SEC and c["n_posts"] > REAL_PREFIX_SIZE]

stats_df = pd.DataFrame([
    {
        "n_cascades": len(real_cascades),
        "n_eligible_for_eval": len(eligible_cascades),
        "avg_posts": np.mean([c["n_posts"] for c in real_cascades]) if real_cascades else 0,
        "median_posts": np.median([c["n_posts"] for c in real_cascades]) if real_cascades else 0,
        "avg_duration_hours": np.mean([c["duration_sec"] for c in real_cascades]) / 3600 if real_cascades else 0,
    }
])

stats_df

In [ ]:
real_eval_df = evaluate_cascades(
    cascades=real_cascades,
    prefix_size=REAL_PREFIX_SIZE,
    total_horizon_sec=REAL_TOTAL_HORIZON_SEC,
    hour_horizon=REAL_HOUR_HORIZON,
    n_simulations=500,
    seed=RANDOM_SEED,
)

real_metrics = pd.DataFrame({
    "metric": [
        "n_evaluated_cascades",
        "evaluation_horizon_hours",
        "RMSE(next_hour_count)",
        "RMSE(final_size_at_horizon)",
        "corr(alpha, final_size_at_horizon)",
    ],
    "value": [
        int(len(real_eval_df)),
        REAL_TOTAL_HORIZON_SEC / 3600,
        safe_rmse(real_eval_df["actual_next_hour"], real_eval_df["pred_next_hour"]),
        safe_rmse(real_eval_df["actual_total"], real_eval_df["pred_total"]),
        safe_corr(real_eval_df["alpha"], real_eval_df["actual_total"]),
    ],
})

real_metrics

In [ ]:
# Визуализация одного реального каскада.
# Берём каскад с максимальным observed final size среди тех, что попали в оценку.

if real_eval_df.empty:
    raise ValueError("После фильтрации и оценки не осталось каскадов. Ослабьте MIN_POSTS_PER_CASCADE или проверьте колонки.")

best_row = real_eval_df.sort_values("actual_total", ascending=False).iloc[0]
example_cluster_id = best_row.get("cluster_id", None)
example_cascade = next(c for c in real_cascades if c["cluster_id"] == example_cluster_id)
example_ts = np.asarray(example_cascade["timestamps"], dtype=float)

full_fit = fit_hawkes_mle(example_ts)
full_grid = np.linspace(0, example_ts[-1], 700)
full_intensity = intensity_curve(
    mu=full_fit["mu"],
    alpha=full_fit["alpha"],
    decay=full_fit["decay"],
    history=example_ts,
    grid=full_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].step(example_ts / 3600, np.arange(1, len(example_ts) + 1), where="post", color="navy")
axes[0].axvline(example_ts[REAL_PREFIX_SIZE - 1] / 3600, color="red", linestyle="--", label=f"prefix N={REAL_PREFIX_SIZE}")
axes[0].set_title(f"Каскад {example_cluster_id}: накопленное число публикаций")
axes[0].set_xlabel("Время, часы")
axes[0].set_ylabel("Число публикаций")
axes[0].legend()

axes[1].plot(full_grid / 3600, full_intensity, color="darkorange")
axes[1].vlines(example_ts / 3600, ymin=0, ymax=full_intensity.max(), color="gray", alpha=0.15)
axes[1].set_title(f"Каскад {example_cluster_id}: оценённая интенсивность")
axes[1].set_xlabel("Время, часы")
axes[1].set_ylabel("Интенсивность")

plt.tight_layout()
plt.show()

In [ ]:
if real_eval_df.empty:
    raise ValueError("Нет оценённых каскадов для построения scatter-графиков.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(real_eval_df["alpha"], real_eval_df["actual_total"], alpha=0.75)
axes[0].set_title("Связь alpha и реального размера каскада")
axes[0].set_xlabel("Оценка alpha по первым N событиям")
axes[0].set_ylabel("Фактический итоговый размер каскада")

axes[1].scatter(real_eval_df["actual_total"], real_eval_df["pred_total"], alpha=0.75)
line_min = min(real_eval_df["actual_total"].min(), real_eval_df["pred_total"].min())
line_max = max(real_eval_df["actual_total"].max(), real_eval_df["pred_total"].max())
axes[1].plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black")
axes[1].set_title("Прогноз итогового размера каскада")
axes[1].set_xlabel("Фактический размер")
axes[1].set_ylabel("Предсказанный размер")

plt.tight_layout()
plt.show()

## Что писать в дипломе по результатам ноутбука

### Для синтетики

Можно формулировать так:

- На синтетически сгенерированных каскадах Hawkes-модель восстанавливает параметры процесса с приемлемой точностью.
- Это подтверждает корректность выбранного подхода и реализации.

### Для реальных данных

Можно формулировать так:

- Для каждого информационного события поток публикаций интерпретируется как самовозбуждающийся точечный процесс.
- Параметр `alpha` характеризует силу каскадного эффекта: чем он выше, тем сильнее одно сообщение стимулирует появление следующих.
- Параметр `mu` отражает фоновую активность, не связанную напрямую с внутренним каскадным размножением.
- Параметр `decay` показывает, как быстро затухает интерес к событию.
- Прогноз по первым `N` сообщениям позволяет оценить дальнейшую динамику обсуждения и ожидаемый размер каскада на фиксированном горизонте наблюдения.

### Ограничения метода

Нужно явно указать ограничения:

- Нужна предварительная кластеризация новостей по событиям.
- Одномерный Hawkes не учитывает тип источника, содержание текста и внешние шоки.
- Для честной оценки нужен фиксированный горизонт прогноза и каскады, наблюдавшиеся не меньше этого горизонта.
- Для очень коротких каскадов параметры оцениваются нестабильно.
- Рекурсивный расчёт likelihood реализован последовательно и не оптимизирован под очень большие каскады.